# Common Settings

In [1]:
from transformers.utils import logging

logging.set_verbosity_error()

# Loading a Text Generation Model

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="mps",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [3]:
# Prompt
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate the output
output = pipe(
    messages,
    do_sample=False
)
print(output[0]["generated_text"])


 Why did the chicken join the band? Because it had the drumsticks!


In [4]:
# Apply prompt template
prompt = pipe.tokenizer.apply_chat_template(
    messages, 
    tokenize=False
)
# End with <|endoftext|>
print(prompt)


<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


In [5]:
# Apply prompt template with add_generation_prompt=true
prompt = pipe.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
# End with <|assistant|>
print(prompt)


<|user|>
Create a funny joke about chickens.<|end|>
<|assistant|>



In [6]:
# Using a high temperature
output = pipe(
    messages,
    do_sample=True,
    temperature=1
)
print(output[0]["generated_text"])


 Why don't chickens use bicycles? Because they're afraid of the road!


In [7]:
# Using a high top_p
output = pipe(
    messages,
    do_sample=True,
    top_p=1,
)
print(output[0]["generated_text"])


 Why don't chickens ever play hide and seek? Because good luck hiding when there's a farmer around to spot your pecky spots!


# Advanced Prompt Engineering

## The Potential Complexity of a Prompt

In [8]:
# Text to summarize which we stole from https://jalammar.github.io/illustrated-transformer/ ;)
text = """In the previous post, we looked at Attention – a ubiquitous method in modern deep learning models. Attention is a concept that helped improve the performance of neural machine translation applications. In this post, we will look at The Transformer – a model that uses attention to boost the speed with which these models can be trained. The Transformer outperforms the Google Neural Machine Translation model in specific tasks. The biggest benefit, however, comes from how The Transformer lends itself to parallelization. It is in fact Google Cloud’s recommendation to use The Transformer as a reference model to use their Cloud TPU offering. So let’s try to break the model apart and look at how it functions.
The Transformer was proposed in the paper Attention is All You Need. A TensorFlow implementation of it is available as a part of the Tensor2Tensor package. Harvard’s NLP group created a guide annotating the paper with PyTorch implementation. In this post, we will attempt to oversimplify things a bit and introduce the concepts one by one to hopefully make it easier to understand to people without in-depth knowledge of the subject matter.
Let’s begin by looking at the model as a single black box. In a machine translation application, it would take a sentence in one language, and output its translation in another.
Popping open that Optimus Prime goodness, we see an encoding component, a decoding component, and connections between them.
The encoding component is a stack of encoders (the paper stacks six of them on top of each other – there’s nothing magical about the number six, one can definitely experiment with other arrangements). The decoding component is a stack of decoders of the same number.
The encoders are all identical in structure (yet they do not share weights). Each one is broken down into two sub-layers:
The encoder’s inputs first flow through a self-attention layer – a layer that helps the encoder look at other words in the input sentence as it encodes a specific word. We’ll look closer at self-attention later in the post.
The outputs of the self-attention layer are fed to a feed-forward neural network. The exact same feed-forward network is independently applied to each position.
The decoder has both those layers, but between them is an attention layer that helps the decoder focus on relevant parts of the input sentence (similar what attention does in seq2seq models).
Now that we’ve seen the major components of the model, let’s start to look at the various vectors/tensors and how they flow between these components to turn the input of a trained model into an output.
As is the case in NLP applications in general, we begin by turning each input word into a vector using an embedding algorithm.
Each word is embedded into a vector of size 512. We'll represent those vectors with these simple boxes.
The embedding only happens in the bottom-most encoder. The abstraction that is common to all the encoders is that they receive a list of vectors each of the size 512 – In the bottom encoder that would be the word embeddings, but in other encoders, it would be the output of the encoder that’s directly below. The size of this list is hyperparameter we can set – basically it would be the length of the longest sentence in our training dataset.
After embedding the words in our input sequence, each of them flows through each of the two layers of the encoder.
Here we begin to see one key property of the Transformer, which is that the word in each position flows through its own path in the encoder. There are dependencies between these paths in the self-attention layer. The feed-forward layer does not have those dependencies, however, and thus the various paths can be executed in parallel while flowing through the feed-forward layer.
Next, we’ll switch up the example to a shorter sentence and we’ll look at what happens in each sub-layer of the encoder.
Now We’re Encoding!
As we’ve mentioned already, an encoder receives a list of vectors as input. It processes this list by passing these vectors into a ‘self-attention’ layer, then into a feed-forward neural network, then sends out the output upwards to the next encoder.
"""

# Prompt components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
text = "MY TEXT TO SUMMARIZE"  # Replace with your own text to summarize
data = f"Text to summarize: {text}"

# The full prompt - remove and add pieces to view its impact on the generated output
query = persona + instruction + context + data_format + audience + tone + data


In [9]:
messages = [
    {"role": "user", "content": query}
]
print(tokenizer.apply_chat_template(messages, tokenize=False))


<|user|>
You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.
Summarize the key findings of the paper provided.
Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.
Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.
The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.
The tone should be professional and clear.
Text to summarize: MY TEXT TO SUMMARIZE<|end|>
<|endoftext|>


In [10]:
# Generate the output
outputs = pipe(messages)
print(outputs[0]["generated_text"])


 [Method Summary]

- The study explores the impact of transfer learning on the efficiency of Large Language Models (LLMs).

- Researchers trained multiple LLMs on a diverse dataset encompassing various languages and domains.

- A comparative analysis was conducted between models pre-trained on massive corpora and those fine-tuned on specific tasks.

- The experiments utilized state-of-the-art evaluation metrics, including perplexity and BLEU score.


[Main Results]

The research findings indicate that transfer learning significantly boosts the performance of LLMs across language and domain barriers. Specifically, LLMs pre-trained on vast and diverse datasets demonstrate superior generalization capabilities compared to those fine-tuned on narrower tasks. The study further corroborates that transfer learning not only enhances the models' fluency and coherence, as measured by perplexity and BLEU scores but also their adaptability and versatility. These results underscore the importance of

## In-Context Learning: Providing Examples

In [11]:
# Use a single example of using the made-up word in a sentence
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))


<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|end|>
<|assistant|>
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


In [12]:
# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])


 In the epic fantasy world of Eldoria, brave warrior Sir Lionel screeged fiercely at the dragon, hoping to vanquish the beast and save the kingdom.


## Chain Prompting: Breaking up the Problem

In [13]:
# Create name and slogan for a product
product_prompt = [
    {"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)


 Name: ChatWiz
Slogan: "Conversing with AI – Your friendly digital wizard!"


In [14]:
# Based on a name and slogan for a product, generate a sales pitch
sales_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]
outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]
print(sales_pitch)


 Discover the magic of modern communication with ChatWiz – where conversing with AI feels like talking to a friendly digital wizard! With ChatWiz, experience seamless and intelligent interactions every time. Unlock the potential of chatbots and elevate your customer service to new heights. Try ChatWiz today and transform the way you converse with technology!


# Reasoning with Generative Models

## Chain-of-Thought: Think Before Answering

In [15]:
# Answering without explicit reasoning
standard_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "11"},
    {"role": "user", "content": "The cafeteria had 25 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

# Run generative model
outputs = pipe(standard_prompt)
print(outputs[0]["generated_text"])


 First, we need to calculate how many apples are left after using 20 for lunch. We start with 25 apples and subtract the 20 used for lunch:


25 apples - 20 apples = 5 apples


Now, we add the 6 new apples that were bought to the remaining 5 apples:


5 apples + 6 apples = 11 apples


Therefore, the cafeteria now has 11 apples.


In [16]:
# Answering with chain-of-thought
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

# Generate the output
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])


 The cafeteria started with 23 apples. They used 20 for lunch, leaving them with 23 - 20 = 3 apples. After buying 6 more, they have 3 + 6 = 9 apples. The answer is 9.


## Zero-shot Chain-of-Thought

In [17]:
# Zero-shot Chain-of-Thought
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]

# Generate the output
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])


 Step 1: Determine the number of apples used
The cafeteria used 20 apples to make lunch.

Step 2: Subtract the used apples from the initial amount
23 apples (initial amount) - 20 apples (used) = 3 apples remaining.

Step 3: Add the newly bought apples to the remaining amount
3 apples (remaining) + 6 apples (bought) = 9 apples in total.

The cafeteria now has 9 apples.


## Tree-of-Thought: Exploring Intermediate Steps

In [18]:
# Zero-shot Chain-of-Thought
zeroshot_tot_prompt = [
    {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realises they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]

# Generate the output
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])


 Expert 1:
Step 1: Calculate the remaining apples after making lunch (23 - 20 = 3 apples remaining).

Expert 2:
Step 1: Calculate the remaining apples after making lunch (23 - 20 = 3 apples remaining).
Step 2: Add the newly bought apples to the remaining apples (3 + 6 = 9 apples remaining).

Expert 3:
Step 1: Subtract the used apples from the initial amount (23 - 20 = 3 apples remaining).
Step 2: Add the newly bought apples to the remaining apples (3 + 6 = 9 apples remaining).

Result Discussion:
Both Expert 2 and Expert 3 arrived at the correct answer, which is 9 apples remaining. Expert 1 also arrived at the correct answer, but they did so by performing the same steps as Expert 2 and Expert 3. The group can agree that the cafeteria now has 9 apples remaining.


# Output Verification

## Providing Examples

In [19]:
# Zero-shot learning: Providing no examples
zeroshot_prompt = [
    {"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]

# Generate the output
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])


 ```json
{
  "name": "Eldrin Sunstrider",
  "class": "Ranger",
  "race": "Elf",
  "level": 10,
  "attributes": {
    "strength": 12,
    "dexterity": 18,
    "constitution": 14,
    "intelligence": 10,
    "wisdom": 16,
    "charisma": 14
  },
  "skills": {
    "archery": 9,
    "traps": 8,
    "survival": 12,
    "stealth": 14,
    "animal handling": 10,
    "crafting": 6
  },
  "equipment": {
    "weapon": "Longbow",
    "armor": "Leather armor",
    "accessories": ["Cloak of Elvenkind", "Gauntlets of the Hawk"],
    "items": ["Water flask", "Small medkit", "


In [20]:
# One-shot learning: Providing an example of the output structure
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:

{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]

# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])


 ```

{

  "description": "A cunning rogue with a mysterious past, skilled in stealth and thievery.",

  "name": "Elira Shadowhand",

  "armor": "Leather Tunic",

  "weapon": "Dual Shortswords"

}

```
